# sosci-pipeline v2.3 — Colab runner
**Ticks / Climate / Sub-Saharan Africa**

Reuse-ready engine. Highlights this version:
- **Multilingual topic modelling by default** — non-English papers cluster by theme, not language (no translation needed).
- **`finalize_corpus`** drops non-tick homonym topics + re-applies the year window → clean analysis-ready corpus.
- **`export`** writes `corpus_explore.xlsx` (per-record) + `analysis_results.xlsx` (aggregate tables).
- **`flows`** emits the corpus-construction flowchart + PRISMA.
- Layers 1–3 now run on the cleaned corpus.


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install + load the package

In [ ]:
!pip install pyarrow pyyaml rapidfuzz openpyxl requests numpy scipy -q
!pip install anthropic openai google-genai scikit-learn -q
!pip install pycountry statsmodels pymannkendall bertopic sentence-transformers umap-learn hdbscan -q

import zipfile, sys, os
ZIP = '/content/drive/MyDrive/sosci-pipeline.zip'   # <-- EDIT to your zip path
zipfile.ZipFile(ZIP).extractall('/content/sosci_pkg')
sys.path.insert(0, '/content/sosci_pkg/sosci-pipeline')
print('loaded sosci_pipeline')

## 3. Build context

In [ ]:
from sosci_pipeline import runner, provenance, export
CONFIG = '/content/sosci_pkg/sosci-pipeline/configs/ticks_climate_ssa.yaml'
ctx = runner.build_ctx(CONFIG, work_subdir='preprocessing_data_v2')
print('files in work dir:')
print(sorted(f for f in os.listdir(ctx.work) if f.endswith(('.parquet','.csv')))[:40])

## 4. API keys (screening / optional translation) 🔑

In [ ]:
from google.colab import userdata
for k in ['ANTHROPIC_API_KEY','OPENAI_API_KEY','GEMINI_API_KEY']:
    try: os.environ[k] = userdata.get(k); print(k,'loaded')
    except Exception: print(k,'NOT set')

## 5. Re-enrich stragglers (optional, repeatable)
Retries only records left unmatched by API rate limits. Re-run with a different
`EMAIL_OVERRIDE` for a fresh OpenAlex polite-pool quota, or after limits reset.

In [ ]:
from sosci_pipeline.reenrich_stragglers import run_reenrich_stragglers
EMAIL_OVERRIDE = None    # e.g. 'other-email@example.com'
MAX_RECORDS    = None    # e.g. 5000 to test; None = all stragglers
run_reenrich_stragglers(ctx, email_override=EMAIL_OVERRIDE, max_records=MAX_RECORDS)

## 6. Tick-relevance (free, rule-based — run before screening)

In [ ]:
runner.run_stage('tick_relevance', ctx)

## 7. Screen — choose which model(s) to run
**Set the flags you want to True.** (Leaving all False is why an earlier run had
nothing to reach consensus on.) Each is independent and resumable.

In [ ]:
from sosci_pipeline import screen
RUN_OPENAI    = True
RUN_ANTHROPIC = True
RUN_GOOGLE    = True
if RUN_OPENAI:
    screen.run_screen(ctx, model_label='openai')

In [ ]:
if RUN_ANTHROPIC:
    screen.run_screen(ctx, model_label='anthropic')

In [ ]:
if RUN_GOOGLE:
    screen.run_screen(ctx, model_label='google')

## 8. Consensus + PRISMA (run only after ≥1 model above finished)

In [ ]:
screen.run_consensus(ctx)
P, path = provenance.write_prisma(ctx.work)
import json; print(json.dumps({k:v for k,v in P.items() if v is not None}, indent=2))

## 9. Topic modelling — MULTILINGUAL
Uses `paraphrase-multilingual-mpnet-base-v2` (set in the config), so French /
Portuguese / Spanish / Russian / Chinese papers cluster by scientific theme.
GPU runtime strongly recommended. Embeddings cache to `topic_embeddings.npy`,
so re-runs that only tweak clustering params don't re-embed.

In [ ]:
runner.run_stage('topic_model', ctx)

## 10. Finalize corpus — drop non-tick noise + fix years
Reads the project taxonomy (`topic_taxonomy.csv`, upload it into the WORK dir),
drops the 8 non-tick homonym topics, re-applies year 2000–2026 on corrected
years, attaches clean labels + theme groups → `master_analysis_ready`.

In [ ]:
# upload projects/ticks_climate_ssa/topic_taxonomy.csv into the WORK dir first, then:
runner.run_stage('finalize_corpus', ctx)

## 11. Analysis layers (run on the cleaned corpus)

In [ ]:
runner.run_stage('layer1_burden', ctx)

In [ ]:
runner.run_stage('layer2_topic_region', ctx)

In [ ]:
runner.run_stage('layer3_climate_niche', ctx)

## 12. Flow diagrams (corpus-construction + PRISMA)

In [ ]:
provenance.build_flows(ctx.work)
print('wrote flow_corpus_construction.json/.mmd and flow_prisma_sr.json')
print(open(os.path.join(ctx.work,'flow_corpus_construction.mmd')).read())

## 13. Export to Excel for exploration
`corpus_explore.xlsx` = curated per-record view (the file after topic modelling,
in Excel). `analysis_results.xlsx` = all aggregate result tables in one workbook.

In [ ]:
export.run_export(ctx, which='both')

## 14. (Optional) Translate non-English text
Only if you want permanent English `title_en`/`abstract_en` (dashboards, or LLM
screening on non-English records). Not needed for topic modelling — the
multilingual embedder already handles languages. Resumable.

In [ ]:
# from sosci_pipeline import analysis_stages
# analysis_stages.run_translate(ctx)